[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S10_pandas_limpieza.ipynb)

# Sesión 10 · pandas: limpieza de datos

**Módulo 3: Pandas** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Detectar y tratar valores nulos con `isna`, `fillna` y `dropna`, incluidos textos como "unknown" y valores centinela.
2. Eliminar filas duplicadas, cambiar tipos con `astype`, limpiar textos con los métodos `.str` y renombrar columnas.
3. Crear columnas nuevas con operaciones, `map`, `apply` y `np.where`.
4. Agrupar valores numéricos en rangos con `pd.cut` y `pd.qcut`.

## 📋 Qué debes saber antes
Sesión 9: cargar un CSV, seleccionar columnas y filtrar filas.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Regla de oro de la limpieza: no toques los datos originales; trabaja siempre sobre una copia o un DataFrame nuevo.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Escribe dos archivos sucios (`clientes_sucios.csv` y `movimientos_sucios.csv`), crea la tabla limpia `clientes_ok` y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, escribe dos archivos CSV sucios y carga los verificadores.
import copy
import csv
import hashlib
import io
import math
import statistics

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

_DISTRITOS = ["Miraflores", "San Isidro", "Surco", "Lince", "Barranco"]

# ---------- Archivo 1: clientes con problemas de calidad ----------
_COLS_C = ["id_cliente", "fecha_alta", "distrito", "edad", "ingreso", "segmento", "canal", "saldo"]
_filas_c = []
for _i in range(40):
    _d = _DISTRITOS[int(rng.integers(0, 5))]
    _variantes = [_d, _d.upper(), _d.lower(), f"  {_d} ", f"{_d.lower()} "]
    _edad = str(int(rng.integers(19, 76)))
    if _i in (4, 17, 30):
        _edad = "-1"                                   # edad desconocida (centinela)
    elif _i in (9, 25):
        _edad = ""
    _ingreso = f"S/ {rng.uniform(1200, 9000):,.2f}"
    if _i == 12:
        _ingreso = ""
    _seg = str(rng.choice(["A", "B", "C"]))
    if _i in (6, 21, 33):
        _seg = "unknown"
    elif _i == 15:
        _seg = ""
    _canal = str(rng.choice(["App", "app ", "AGENCIA", "agencia", " Web", "web"]))
    _saldo = f"{rng.uniform(0, 20000):.2f}"
    if _i in (11, 28):
        _saldo = "999999"                              # saldo no informado (centinela)
    _filas_c.append([str(5001 + _i), f"2025-{int(rng.integers(1, 13)):02d}-{int(rng.integers(1, 29)):02d}",
                     _variantes[int(rng.integers(0, 5))], _edad, _ingreso, _seg, _canal, _saldo])
_filas_c += [list(_filas_c[2]), list(_filas_c[7]), list(_filas_c[19])]     # filas duplicadas
_buf = io.StringIO()
csv.writer(_buf, lineterminator="\n").writerows([_COLS_C] + _filas_c)
_TXT_C = _buf.getvalue()
with open("clientes_sucios.csv", "w", encoding="utf-8") as _f:
    _f.write(_TXT_C)

# ---------- Tabla limpia para crear columnas ----------
_n = 30
clientes_ok = pd.DataFrame({
    "id": np.arange(6001, 6001 + _n),
    "distrito": rng.choice(_DISTRITOS, _n),
    "edad": rng.integers(19, 76, _n),
    "ingreso": np.round(rng.uniform(1500, 9000, _n), 2),
    "gasto": np.round(rng.uniform(800, 6000, _n), 2),
    "segmento": rng.choice(["A", "B", "C"], _n),
})
clientes_ok.loc[7, "segmento"] = "D"                   # un segmento nuevo, sin nombre asignado

# ---------- Archivo 2: movimientos bancarios sucios ----------
_COLS_M = ["id_mov", "fecha", "cliente", "tipo", "monto", "canal", "comision"]
_filas_m = []
for _i in range(30):
    _t = str(rng.choice(["deposito", "retiro", "pago"]))
    _tipo = [_t, _t.upper(), _t.title() + " ", " " + _t][int(rng.integers(0, 4))]
    _m = rng.uniform(200, 5000) if _t == "deposito" else -rng.uniform(20, 3000)
    _monto = "N/A" if _i in (5, 18) else f"S/ {_m:,.2f}"
    _canal = "unknown" if _i in (3, 11, 24) else str(rng.choice(["app", "agencia", "cajero"]))
    _com = "-1" if _i in (7, 20) else f"{rng.uniform(0, 8):.2f}"
    _filas_m.append([str(7001 + _i), f"2026-09-{int(rng.integers(1, 31)):02d}", f"C{int(rng.integers(1, 13)):02d}",
                     _tipo, _monto, _canal, _com])
_filas_m += [list(_filas_m[1]), list(_filas_m[14])]
_buf = io.StringIO()
csv.writer(_buf, lineterminator="\n").writerows([_COLS_M] + _filas_m)
_TXT_M = _buf.getvalue()
with open("movimientos_sucios.csv", "w", encoding="utf-8") as _f:
    _f.write(_TXT_M)

_D = copy.deepcopy({"clientes_ok": clientes_ok})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _filas(txt):
    return list(csv.reader(io.StringIO(txt)))[1:]


def _num(x):
    try:
        return float(x)
    except ValueError:
        return None


def _entero_o_decimal(x):
    v = _num(x)
    return None if v is None else (int(v) if v == int(v) and "." not in x else v)


def _dinero(x):
    return None if x in ("", "N/A") else float(x.replace("S/", "").replace(",", "").strip())


def _sin_duplicados(filas):
    vistas, salida = set(), []
    for f in filas:
        if tuple(f) not in vistas:
            vistas.add(tuple(f))
            salida.append(f)
    return salida


def _clientes_ref():
    """clientes leído con na_values=["unknown"] y centinelas convertidos: filas como listas de Python."""
    salida = []
    for f in _filas(_TXT_C):
        edad = _num(f[3])
        saldo = _num(f[7])
        salida.append([int(f[0]), f[1], f[2], None if edad in (None, -1) else edad, f[4] or None,
                       None if f[5] in ("", "unknown") else f[5], f[6], None if saldo == 999999 else saldo])
    return salida


def _tabla_ref():
    t = _D["clientes_ok"]
    filas = []
    for _, f in t.iterrows():
        ahorro = f["ingreso"] - f["gasto"]
        nombre = {"A": "Premium", "B": "Estándar", "C": "Básico"}.get(f["segmento"])
        filas.append([f["id"], f["distrito"], f["edad"], f["ingreso"], f["gasto"], f["segmento"], ahorro, nombre,
                      "alto" if ahorro > 1000 else "bajo", f["distrito"][:3].upper()])
    return filas


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    filas = _filas(_TXT_C)
    crudos = [[x == "" for x in f] for f in filas]
    _ser(r, "nulos_crudo", [sum(c[j] for c in crudos) for j in range(8)], "cuenta los nulos de cada columna de `crudo`", indice=_COLS_C)
    ref = _clientes_ref()
    cols = ["id_cliente", "fecha_alta", "distrito", "edad", "ingreso", "segmento", "canal", "saldo"]
    _df(r, "clientes", cols, ref, "lee con `na_values` para \"unknown\" y convierte en NaN los centinelas -1 (edad) y 999999 (saldo)")
    _ser(r, "nulos", [sum(f[j] is None for f in ref) for j in range(8)], "cuenta los nulos de cada columna de `clientes`", indice=cols)
    con_seg = [f for f in ref if f[5] is not None]
    _df(r, "sin_nulos_segmento", cols, con_seg, "quita solo las filas sin segmento")
    edades = [f[3] for f in ref if f[3] is not None]
    med = statistics.median(edades)
    _ser(r, "edad_rellena", [med if f[3] is None else f[3] for f in ref], "rellena los NaN de edad con la mediana de la edad")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_media_nan": "31aaae3c5d4ba3268439fabb38e6cc6d8ce216ec1e104aedd8355912f3628b6d",
        "pred_suma_nan": "399afcbd9cfc35e6944d699c5f5b3cbbcd510a3a44f8f08e8172055ba1a27cba",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    filas = _filas(_TXT_C)
    unicas = _sin_duplicados(filas)
    _esc(r, "n_duplicados", len(filas) - len(unicas), "cuenta las filas repetidas (sin contar la primera aparición)")
    x = r.var("sin_dup")
    if x is not _FALTA:
        if not isinstance(x, pd.DataFrame):
            r.mal("`sin_dup` debería ser un DataFrame.")
        elif len(x) != len(unicas) or list(x.columns) != _COLS_C:
            r.mal(f"`sin_dup` tiene {len(x)} filas y se esperaban {len(unicas)}, con todas las columnas.")
        else:
            r.ok("`sin_dup` es correcto.")
    _ser(r, "distrito_limpio", [f[2].strip().title() for f in filas], "sin espacios a los lados y con mayúscula inicial en cada palabra")
    _ser(r, "canal_limpio", [f[6].strip().lower() for f in filas], "sin espacios a los lados y en minúsculas")
    _ser(r, "ingreso_num", [_dinero(f[4]) for f in filas], "quita \"S/\", las comas de miles y los espacios, y convierte a float")
    x = r.var("renombrado")
    if x is not _FALTA:
        esperado = ["id", "alta"] + _COLS_C[2:]
        if isinstance(x, pd.DataFrame) and [str(c) for c in x.columns] == esperado and len(x) == len(filas):
            r.ok("`renombrado` es correcto.")
        else:
            r.mal("`renombrado` debería ser `crudo` con id_cliente → id y fecha_alta → alta, sin tocar las demás columnas.")
    x = globals().get("ids_texto")
    if isinstance(x, pd.Series) and len(x) and not all(isinstance(v, str) for v in x.tolist()):
        r.mal("Los valores de `ids_texto` deberían ser textos: usa `astype(str)`.")
    else:
        _ser(r, "ids_texto", [f[0] for f in filas], "la columna id_cliente convertida a texto")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_title": "d4a5e5e0f8bc741829dd6c0c0b60e755cbc0c33df6cf7c47ceea98327d5fd3b8",
        "pred_sin_dup": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    cols = ["id", "distrito", "edad", "ingreso", "gasto", "segmento", "ahorro", "segmento_nombre", "tipo_ahorrador", "cod_distrito"]
    ref = _tabla_ref()
    _df(r, "tabla", cols, ref, "revisa las cuatro columnas nuevas y su orden", tol=1e-6)
    _esc(r, "n_sin_nombre", sum(f[7] is None for f in ref), "cuenta los NaN de segmento_nombre")
    _sin_cambios_df(r, "clientes_ok")
    r.fin()


def _cut(x, bordes, etiquetas):
    for a, b, e in zip(bordes, bordes[1:], etiquetas):
        if a < x <= b:
            return e
    return None


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    t = _D["clientes_ok"]
    edades = [int(x) for x in t["edad"].tolist()]
    rangos = [_cut(x, [17, 29, 44, 59, 100], ["18-29", "30-44", "45-59", "60+"]) for x in edades]
    _ser(r, "rango_edad", rangos, "usa los bordes 17, 29, 44, 59 y 100 con sus cuatro etiquetas")
    ingresos = [float(x) for x in t["ingreso"].tolist()]
    q = statistics.quantiles(ingresos, n=4, method="inclusive")
    bordes = [min(ingresos) - 1] + q + [max(ingresos)]
    _ser(r, "cuartil_ingreso", [_cut(x, bordes, ["Q1", "Q2", "Q3", "Q4"]) for x in ingresos],
         "cuatro grupos con la misma cantidad de clientes, etiquetados Q1 a Q4")
    _esc(r, "n_jovenes", rangos.count("18-29"), "cuenta los clientes del rango 18-29")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_fuera": "7cd2a33d8476047b3755295f8b4747a3baea572e22f0e24d21505be7bc3c9844",
        "pred_borde": "71fdf9965cf0a0f409974a667c3bdff6932f80d3cde7c7800028c039238bdd08",
    })
    r.fin()


def _reto_ref():
    filas = _filas(_TXT_M)
    unicas = _sin_duplicados(filas)
    salida = []
    for f in unicas:
        monto = _dinero(f[4])
        if monto is None:
            continue
        com = float(f[6])
        a = abs(monto)
        tramo = _cut(a, [0, 100, 500, 2000, math.inf], ["bajo", "medio", "alto", "muy alto"])
        salida.append([int(f[0]), f[1], f[2], f[3].strip().lower(), monto, None if f[5] == "unknown" else f[5],
                       None if com == -1 else com, monto < 0, tramo])
    return len(filas), salida


def check_reto():
    r = _Revision("Reto final")
    n_crudo, ref = _reto_ref()
    cols = ["id", "fecha", "cliente", "tipo", "monto", "canal", "comision", "es_egreso", "tramo"]
    _df(r, "movs_limpios", cols, ref, "revisa cada paso de la lista, en orden", tol=0.0051)
    _esc(r, "n_eliminadas", n_crudo - len(ref), "filas del archivo original menos filas de `movs_limpios`")
    _esc(r, "pct_canal_nulo", round(sum(f[5] is None for f in ref) * 100 / len(ref), 1), "porcentaje de canal nulo en `movs_limpios`, con 1 decimal", tol=0.051)
    _esc(r, "total_egresos", round(math.fsum(f[4] for f in ref if f[4] < 0), 2), "suma de los montos negativos, con 2 decimales", tol=0.0051)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    filas = _filas(_TXT_C)
    x = r.var("partes")
    if x is not _FALTA:
        if isinstance(x, pd.DataFrame) and x.shape == (len(filas), 3) and x.iloc[0].tolist() == filas[0][1].split("-"):
            r.ok("`partes` es correcto.")
        else:
            r.mal(f"`partes` debería ser un DataFrame de {len(filas)} filas y 3 columnas (año, mes, día), con `expand=True`.")
    _ser(r, "anio_alta", [int(f[1][:4]) for f in filas], "la primera parte de la fecha, como número entero")
    x = r.var("distrito_cat")
    if x is not _FALTA:
        if isinstance(x, pd.Series) and isinstance(x.dtype, pd.CategoricalDtype) and len(x.cat.categories) == 5:
            r.ok("`distrito_cat` es una columna de categorías con los 5 distritos.")
        else:
            r.mal("`distrito_cat` debería ser `distrito_limpio` convertido al tipo category, con 5 categorías.")
    r.fin()


print("✅ Setup listo. Archivos CSV escritos y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales. Mira con atención los problemas de los archivos: espacios, mayúsculas mezcladas, montos escritos como texto, "unknown", valores imposibles y filas repetidas.

In [ ]:
for archivo in ["clientes_sucios.csv", "movimientos_sucios.csv"]:
    with open(archivo, encoding="utf-8") as f:
        print(f"--- {archivo}")
        for _ in range(8):
            print(f.readline().rstrip())
print()
print("clientes_ok:")
print(clientes_ok.head(8))

---
## 1. Valores nulos

### 📘 Concepto
En pandas, un dato faltante es `NaN` (o `None`). Al leer un CSV, pandas convierte en `NaN` los campos vacíos y textos como `"NA"`, `"N/A"`, `"NaN"` o `"null"`. Otros marcadores no los reconoce solo:
- **Textos** como `"unknown"` o `"sin dato"`: indícalos al leer con `pd.read_csv(ruta, na_values=["unknown"])`.
- **Valores centinela** numéricos como `-1` o `999999`: conviértelos después con `df["col"].replace(-1, np.nan)`.

| Código | Qué hace |
|---|---|
| `df.isna()` | `True` donde hay nulo; `df.isna().sum()` los cuenta por columna |
| `df.dropna()` | quita las filas con algún nulo; con `subset=["col"]`, solo mira esas columnas |
| `df["col"].fillna(valor)` | reemplaza los nulos de la columna por `valor` |

A diferencia de NumPy, `sum`, `mean`, `median` y compañía de pandas **ignoran** los `NaN` por defecto.

Para cambiar una columna, asígnala de vuelta: `df["col"] = df["col"].replace(...)`.

In [ ]:
with open("mini_ej.csv", "w") as f:
    f.write("cliente,edad,segmento\nC1,34,A\nC2,-1,sin dato\nC3,,B\n")

mini_ej = pd.read_csv("mini_ej.csv", na_values=["sin dato"])
mini_ej["edad"] = mini_ej["edad"].replace(-1, np.nan)
print(mini_ej)
print(mini_ej.isna().sum())
print(mini_ej["edad"].mean())                      # ignora los NaN
print(mini_ej.dropna(subset=["segmento"]))
print(mini_ej["edad"].fillna(mini_ej["edad"].median()))

### ✍️ Tu turno · Ejercicio 1: contar y tratar nulos
**Parte A.** Hazlo todo en esta celda, para poder volver a ejecutarla desde cero:
1. `crudo`: carga `clientes_sucios.csv` tal cual, y `nulos_crudo`: los nulos por columna de `crudo`.
2. `clientes`: carga el mismo archivo indicando que `"unknown"` es un nulo. Luego convierte en `NaN` los centinelas: `-1` en edad y `999999` en saldo.
3. `nulos`: los nulos por columna de `clientes`.
4. `sin_nulos_segmento`: `clientes` sin las filas que no tienen segmento.
5. `edad_rellena`: la columna edad de `clientes` con los nulos reemplazados por la **mediana** de la edad.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_media_nan` | `pd.Series([1, np.nan, 3]).mean()` | número o `"nan"` |
| `pred_suma_nan` | `pd.Series([np.nan, np.nan]).sum()` | número o `"nan"` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`crudo` y `clientes` son dos lecturas del mismo archivo: la segunda con `na_values=["unknown"]`. Compara `nulos_crudo` con `nulos` para ver qué cambió.
</details>

<details><summary>💡 Pista 2</summary>

Los centinelas se convierten columna por columna: `clientes["edad"] = clientes["edad"].replace(-1, np.nan)`. Para `edad_rellena`, calcula primero la mediana y pásala a `fillna`.
</details>

---
## 2. Duplicados, tipos, textos y nombres

### 📘 Concepto
| Código | Qué hace |
|---|---|
| `df.duplicated()` | `True` en cada fila que repite una anterior; `.sum()` las cuenta |
| `df.drop_duplicates()` | quita las repetidas y deja la primera aparición |
| `df["col"].astype(float)` | convierte el tipo (`int`, `float`, `str`, `"category"`...); falla si algún valor no se puede convertir |
| `df.rename(columns={"viejo": "nuevo"})` | renombra columnas; las demás quedan igual |

Los métodos de texto se usan sobre una columna con el prefijo **`.str`**: `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.title()` (mayúscula inicial en cada palabra) y `.str.replace(viejo, nuevo)`. Se pueden encadenar, y los nulos se quedan como nulos.

Un número guardado como texto (`"S/ 1,250.00"`) no se puede sumar: primero se limpia con `.str.replace` y luego se convierte con `astype(float)`.

In [ ]:
precios_ej = pd.DataFrame({"producto": [" Polo", "JEAN ", "polo", " Polo"],
                           "precio": ["S/ 39.90", "S/ 1,129.90", "S/ 45.00", "S/ 39.90"]})
print(precios_ej.duplicated().sum())            # la última fila repite la primera
print(precios_ej.drop_duplicates())
print(precios_ej["producto"].str.strip().str.lower())
print(precios_ej["precio"].str.replace("S/", "").str.replace(",", "").astype(float))
print(precios_ej.rename(columns={"precio": "precio_txt"}).columns.tolist())

### ✍️ Tu turno · Ejercicio 2: estandarizar clientes
**Parte A.** Trabaja sobre `crudo` sin modificarlo (cada resultado en una variable nueva):
1. `n_duplicados`: cuántas filas repetidas hay, y `sin_dup`: `crudo` sin ellas.
2. `distrito_limpio`: la columna distrito sin espacios a los lados y con mayúscula inicial en cada palabra (por ejemplo, `"San Isidro"`).
3. `canal_limpio`: la columna canal sin espacios a los lados y en minúsculas.
4. `ingreso_num`: la columna ingreso como número: quita `"S/"`, las comas de miles y los espacios, y convierte a `float`.
5. `renombrado`: `crudo` con id_cliente renombrada a `id` y fecha_alta a `alta`.
6. `ids_texto`: la columna id_cliente convertida a texto.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_title` | `"SAN ISIDRO".title()` | texto |
| `pred_sin_dup` | `len(pd.Series([1, 1, 2]).drop_duplicates())` | número |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Cada punto es una línea que empieza con `crudo[...]` o `crudo.` y encadena métodos. Si `astype(float)` da error, imprime los valores que no pudiste limpiar.
</details>

<details><summary>💡 Pista 2</summary>

Para `ingreso_num`: `.str.replace("S/", "")`, luego `.str.replace(",", "")`, luego `.str.strip()` y al final `.astype(float)`.
</details>

---
## 3. Crear columnas: operaciones, `map`, `apply` y `np.where`

### 📘 Concepto
Una columna nueva se crea asignándola: `df["nueva"] = ...`.
- **Operación entre columnas**: `df["margen"] = df["precio"] - df["costo"]`.
- **`map` con un diccionario**: traduce cada valor. Los valores que no están en el diccionario quedan como `NaN` (revísalo siempre).
- **`apply` con una función**: aplica la función a cada valor de la columna. Úsalo cuando no hay una operación vectorizada que sirva.
- **`np.where(condición, si, no)`**: elige entre dos valores, como en NumPy.

Para no cambiar un DataFrame que quieres conservar, trabaja sobre `df.copy()`.

In [ ]:
pedidos_ej = pd.DataFrame({"producto": ["polo", "jean", "gorra"], "precio": [39.9, 129.9, 25.0],
                           "costo": [18.0, 70.0, 20.0], "talla": ["M", "S", "XL"]})
pedidos_ej["margen"] = pedidos_ej["precio"] - pedidos_ej["costo"]
pedidos_ej["talla_nombre"] = pedidos_ej["talla"].map({"S": "pequeña", "M": "mediana", "L": "grande"})
pedidos_ej["codigo"] = pedidos_ej["producto"].apply(lambda p: p[:2].upper())
pedidos_ej["rentable"] = np.where(pedidos_ej["margen"] > 20, "sí", "no")
print(pedidos_ej)                           # XL no estaba en el diccionario: NaN

### ✍️ Tu turno · Ejercicio 3: enriquecer la tabla de clientes
Crea `tabla` como copia de `clientes_ok` y agrégale, en este orden:
1. `ahorro`: ingreso menos gasto.
2. `segmento_nombre`: el segmento traducido con `map`: A → `"Premium"`, B → `"Estándar"`, C → `"Básico"`.
3. `tipo_ahorrador`: `"alto"` si el ahorro es mayor que 1000 y `"bajo"` si no, con `np.where`.
4. `cod_distrito`: las 3 primeras letras del distrito en mayúsculas, con `apply` y una `lambda`.

Luego, `n_sin_nombre`: cuántos clientes quedaron sin `segmento_nombre` (¿por qué?). `clientes_ok` no debe cambiar.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Empieza con `tabla = clientes_ok.copy()`. Cada columna nueva es una línea `tabla["nombre"] = ...`.
</details>

<details><summary>💡 Pista 2</summary>

La lambda de `cod_distrito` recibe un texto: `lambda d: d[:3].upper()`. Para `n_sin_nombre`, usa `.isna().sum()` sobre la columna nueva.
</details>

---
## 4. Rangos con `pd.cut` y `pd.qcut`

### 📘 Concepto
Convertir un número en un rango ("30-44 años", "ingreso alto") facilita resumir y comunicar.
- **`pd.cut(serie, bins=bordes, labels=etiquetas)`**: tú eliges los bordes. Cada intervalo **incluye su borde derecho** y excluye el izquierdo: con bordes `[17, 29, 44]`, el 29 cae en el primer rango y el 30 en el segundo. Los valores fuera de los bordes quedan como `NaN`.
- **`pd.qcut(serie, q=4, labels=etiquetas)`**: pandas elige los bordes para que cada grupo tenga **la misma cantidad** de valores (aquí, cuartiles).

Las dos devuelven una columna de tipo categoría. Necesitas una etiqueta menos que bordes.

In [ ]:
tickets_ej = pd.Series([12, 45, 80, 150, 300, 20, 95, 60])
print(pd.cut(tickets_ej, bins=[0, 50, 100, 1000], labels=["bajo", "medio", "alto"]))
print(pd.qcut(tickets_ej, q=2, labels=["mitad baja", "mitad alta"]))
print(pd.cut(pd.Series([50, 51]), bins=[0, 50, 100]))     # 50 cae en el primer intervalo

### ✍️ Tu turno · Ejercicio 4: rangos de edad y cuartiles de ingreso
**Parte A.** Con la columna edad y la columna ingreso de `tabla`:
1. `rango_edad`: la edad en rangos con bordes `[17, 29, 44, 59, 100]` y etiquetas `"18-29"`, `"30-44"`, `"45-59"` y `"60+"`.
2. `cuartil_ingreso`: el ingreso en 4 grupos del mismo tamaño, con etiquetas `"Q1"` a `"Q4"`.
3. `n_jovenes`: cuántos clientes hay en el rango `"18-29"`.

**Parte B.** Predice **sin ejecutar** (escribe la etiqueta como texto, o `"nan"`):

| Variable | Pregunta |
|---|---|
| `pred_fuera` | `pd.cut(pd.Series([15]), bins=[17, 29, 44], labels=["a", "b"])[0]` |
| `pred_borde` | `pd.cut(pd.Series([29]), bins=[17, 29, 44], labels=["a", "b"])[0]` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`pd.cut` recibe la columna, `bins` y `labels`. `pd.qcut` recibe la columna, `q` y `labels`.
</details>

<details><summary>💡 Pista 2</summary>

Para contar: `(rango_edad == "18-29").sum()`. En la parte B, recuerda qué borde incluye cada intervalo.
</details>

---
## 🏋️ Reto final: limpiar los movimientos bancarios
Construye `movs_limpios` a partir de `movimientos_sucios.csv` siguiendo estos pasos, en orden:
1. Carga el archivo indicando que `"unknown"` es un nulo (`"N/A"` pandas ya lo reconoce).
2. Quita las filas duplicadas.
3. `tipo`: sin espacios a los lados y en minúsculas.
4. `monto`: como número (quita `"S/"`, las comas de miles y los espacios, y convierte a `float`).
5. `comision`: el centinela `-1` como nulo.
6. Quita las filas sin monto.
7. Agrega `es_egreso`: `True` si el monto es negativo.
8. Agrega `tramo`: el **valor absoluto** del monto (investiga el método `.abs()`) en rangos con bordes `[0, 100, 500, 2000, np.inf]` y etiquetas `"bajo"`, `"medio"`, `"alto"` y `"muy alto"`.
9. Renombra id_mov a `id`.

Luego calcula:
- `n_eliminadas`: cuántas filas del archivo original no llegaron a `movs_limpios`.
- `pct_canal_nulo`: el porcentaje de filas de `movs_limpios` sin canal, con 1 decimal.
- `total_egresos`: la suma de los montos negativos, con 2 decimales.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Escribe una línea por paso, reasignando `movs_limpios` cada vez. Imprime `movs_limpios.head()` y `movs_limpios.dtypes` entre pasos para ver cómo cambia.
</details>

<details><summary>💡 Pista 2</summary>

Para `n_eliminadas` necesitas el número de filas del archivo original: léelo aparte o guarda `len(...)` justo después de cargarlo. El paso 6 es `dropna(subset=["monto"])`.
</details>

---
## 🚀 Nivel pro (opcional)
1. `partes`: la columna fecha_alta de `crudo` dividida por `"-"` en tres columnas (año, mes y día). Investiga el parámetro `expand=True` de `.str.split`.
2. `anio_alta`: la primera de esas columnas convertida a entero.
3. `distrito_cat`: `distrito_limpio` convertido al tipo `"category"`. Mira `distrito_cat.cat.categories`: ¿cuántos distritos distintos quedaron después de limpiar?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Contar los nulos de cada columna y decidir entre `dropna` y `fillna`.
- [ ] Convertir en nulos los textos como "unknown" al leer y los valores centinela después.
- [ ] Explicar por qué pandas y NumPy dan resultados distintos al promediar con nulos.
- [ ] Encontrar y quitar filas duplicadas.
- [ ] Limpiar textos con `.str` y convertir un número escrito como texto a `float`.
- [ ] Renombrar columnas y cambiar tipos con `astype`.
- [ ] Crear columnas con operaciones, `map`, `apply` y `np.where`, y detectar los `NaN` que deja `map`.
- [ ] Explicar la diferencia entre `pd.cut` y `pd.qcut`, y qué borde incluye cada intervalo.

**Próxima sesión (S11):** agrupar y resumir con `groupby`, `value_counts`, `crosstab` y `pivot_table`.